In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from metrik import ing_hubs_datathon_metric

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RANDOM_STATE = 42

def eval_ing(y_true, y_prob, label=""):
    auc = roc_auc_score(y_true, y_prob)
    gini = 2*auc - 1
    ing  = ing_hubs_datathon_metric(y_true, y_prob)
    print(f"{label}AUC: {auc:.4f} | Gini: {gini:.4f} | ING: {ing:.4f}")
    return ing, auc, gini


In [ ]:
TRAIN_PATH = "../data/train_features_fast.csv"
TEST_PATH  = "../data/test_features_fast.csv"

TARGET = "churn"
DROP = ["cust_id", "ref_date", "date"]  # yoksa ignore edilip düşer

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Kategorik kolonlar (iş sorunlarındaki 'M' burada -> mutlaka kategorik tutulmalı)
cat_cols = [
    "gender","province","religion","work_type","work_sector",
    "NEW_TENURE_GROUP","NEW_AGE_GROUP"
]

# Numerik kolonlar (mevcut özellik listesinden uyarladık; varsa ekle/çıkar)
num_cols = [
    "age","tenure",
    "NEW_TOTAL_TX_AMT_mean","NEW_TOTAL_TX_AMT_sum","NEW_TOTAL_TX_AMT_var",
    "NEW_TOTAL_TX_CNT_mean","NEW_TOTAL_TX_CNT_sum",
    "active_product_category_nbr_mean",
    "NEW_EFT_TO_CC_RATIO_mean","NEW_ACTIVITY_FLAG_mean",
    "avg_tx_amt_last6m"
]

# Hızlı temizlik: numeriklerde string kaçaklarını NaN'a çevir
for c in num_cols:
    if c in train_df.columns:
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    if c in test_df.columns:
        test_df[c]  = pd.to_numeric(test_df[c], errors="coerce")

# Kategorikleri category dtype yap
for c in cat_cols:
    if c in train_df.columns: train_df[c] = train_df[c].astype("category")
    if c in test_df.columns:  test_df[c]  = test_df[c].astype("category")


In [ ]:
# Numerik KNN imputasyon: age & tenure (istersen diğer num'ları da ekleyebilirsin)
knn_num_cols = [c for c in ["age","tenure"] if c in train_df.columns]

if knn_num_cols:
    knn_imputer = KNNImputer(n_neighbors=5, weights="distance")
    train_df[knn_num_cols] = knn_imputer.fit_transform(train_df[knn_num_cols])
    test_df[knn_num_cols]  = knn_imputer.transform(test_df[knn_num_cols])


In [ ]:
# Kategoriklerde Unknown'ları KNN ile doldurmak için:
#  - train+test birlikte LabelEncode
#  - Tüm cat kolonlarını birlikte KNN impute
#  - En yakın sınıfa (round/clip) ve inverse_transform geri çevir

cat_cols_present = [c for c in cat_cols if c in train_df.columns]

# Eğer kategorik sütun varsa
if cat_cols_present:
    encoders = {}
    # Encode all cat cols jointly to keep mapping consistent
    cat_encoded_train = pd.DataFrame(index=train_df.index)
    cat_encoded_test  = pd.DataFrame(index=test_df.index)

    for c in cat_cols_present:
        le = LabelEncoder()
        combined = pd.concat([train_df[c].astype(str), test_df[c].astype(str)], axis=0)
        le.fit(combined.fillna("Unknown"))
        encoders[c] = le
        cat_encoded_train[c] = le.transform(train_df[c].astype(str).fillna("Unknown"))
        cat_encoded_test[c]  = le.transform(test_df[c].astype(str).fillna("Unknown"))

    # KNN imputasyon (kategorik kodlar üzerinde)
    knn_imputer_cat = KNNImputer(n_neighbors=5)
    imputed_train = knn_imputer_cat.fit_transform(cat_encoded_train)
    imputed_test  = knn_imputer_cat.transform(cat_encoded_test)

    # Round & clip to valid class range
    imputed_train = np.rint(imputed_train).astype(int)
    imputed_test  = np.rint(imputed_test).astype(int)
    for j, c in enumerate(cat_cols_present):
        classes = np.arange(len(encoders[c].classes_))
        imputed_train[:, j] = np.clip(imputed_train[:, j], classes.min(), classes.max())
        imputed_test[:, j]  = np.clip(imputed_test[:, j], classes.min(), classes.max())

    # Inverse transform back to original labels
    for idx, c in enumerate(cat_cols_present):
        train_df[c] = encoders[c].inverse_transform(imputed_train[:, idx])
        test_df[c]  = encoders[c].inverse_transform(imputed_test[:, idx])

    # Yeniden kategori dtype
    for c in cat_cols_present:
        train_df[c] = train_df[c].astype("category")
        test_df[c]  = test_df[c].astype("category")


In [ ]:
# Yeni özellikler (mevcut kolonlara göre güvenli hesaplama)
if "NEW_TOTAL_TX_CNT_sum" in train_df.columns and "tenure" in train_df.columns:
    train_df["TX_DENSITY"] = train_df["NEW_TOTAL_TX_CNT_sum"] / (train_df["tenure"] + 1)
    test_df["TX_DENSITY"]  = test_df["NEW_TOTAL_TX_CNT_sum"]  / (test_df["tenure"] + 1)

if "active_product_category_nbr_mean" in train_df.columns and "NEW_TOTAL_TX_CNT_mean" in train_df.columns:
    train_df["AVG_PRODUCT_USE"] = train_df["active_product_category_nbr_mean"] / (train_df["NEW_TOTAL_TX_CNT_mean"] + 1)
    test_df["AVG_PRODUCT_USE"]  = test_df["active_product_category_nbr_mean"]  / (test_df["NEW_TOTAL_TX_CNT_mean"] + 1)

if "NEW_EFT_TO_CC_RATIO_mean" in train_df.columns:
    train_df["CC_DOMINANT"] = (train_df["NEW_EFT_TO_CC_RATIO_mean"] < 1).astype(int)
    test_df["CC_DOMINANT"]  = (test_df["NEW_EFT_TO_CC_RATIO_mean"] < 1).astype(int)

# Ölçeklenecek numerikler
scale_cols = [c for c in [
    "age","tenure",
    "NEW_TOTAL_TX_AMT_mean","NEW_TOTAL_TX_AMT_sum","NEW_TOTAL_TX_AMT_var",
    "NEW_TOTAL_TX_CNT_mean","NEW_TOTAL_TX_CNT_sum",
    "active_product_category_nbr_mean",
    "NEW_EFT_TO_CC_RATIO_mean","NEW_ACTIVITY_FLAG_mean",
    "avg_tx_amt_last6m",
    "TX_DENSITY","AVG_PRODUCT_USE"
] if c in train_df.columns]

# Fatal NaN kalmasın
if scale_cols:
    imp_median = SimpleImputer(strategy="median")
    train_df[scale_cols] = imp_median.fit_transform(train_df[scale_cols])
    test_df[scale_cols]  = imp_median.transform(test_df[scale_cols])

    scaler = RobustScaler()
    train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
    test_df[scale_cols]  = scaler.transform(test_df[scale_cols])


In [ ]:
# Model giriş-çıkış
X = train_df.drop(columns=[TARGET] + DROP, errors="ignore")
y = train_df[TARGET]

# Kategorikler X bazında dursun
cat_cols_in_X = [c for c in cat_cols if c in X.columns]
cat_idx = [X.columns.get_loc(c) for c in cat_cols_in_X]   # 🔥 HATA KAYNAĞI BURASIYDI — X'e göre indeks!

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Sağlama: cat_idx valid için de aynı sırayı temsil eder (X_train, X_valid aynı kolon sırası)
assert list(X_train.columns) == list(X_valid.columns), "Train/Valid kolon sırası uyuşmuyor!"


In [ ]:
# Param alanı: 3x3x4x3x3x1 ≈ 108; ama RandomSampler ile 48 deneme alıyoruz.
param_space = {
    "depth":               [6, 7, 8],
    "learning_rate":       [0.03, 0.035, 0.04],
    "l2_leaf_reg":         [2, 3, 5, 7],
    "random_strength":     [0.5, 1.0, 1.5],
    "bagging_temperature": [0.5, 1.0, 1.5],
    "grow_policy":         ["Depthwise"],   # istersen ["Depthwise","Lossguide"]
    "border_count":        [128],           # istersen [128, 254]
}

N_TRIALS = 48
samples = list(ParameterSampler(param_space, n_iter=N_TRIALS, random_state=RANDOM_STATE))

best = {"score": -1, "params": None, "model": None}

for i, p in enumerate(samples, 1):
    cb = CatBoostClassifier(
        task_type="GPU", devices="0",
        iterations=1500,
        od_type="Iter", od_wait=150,
        loss_function="Logloss", eval_metric="AUC",
        random_seed=RANDOM_STATE, verbose=False,
        **p
    )
    cb.fit(X_train, y_train, eval_set=(X_valid, y_valid), cat_features=cat_idx)
    preds = cb.predict_proba(X_valid)[:, 1]
    score = ing_hubs_datathon_metric(y_valid, preds)
    if i % 5 == 0:
        print(f"[{i}/{N_TRIALS}] ING={score:.5f} | params={p}")
    if score > best["score"]:
        best = {"score": score, "params": p, "model": cb}

print("\n🏆 BEST (valid):", best["score"])
print("PARAMS:", best["params"])


In [ ]:
pos = int(y_train.sum()); neg = len(y_train) - pos
w_pos = float(neg / pos)  # ~6-7 civarı çıkar

p = best["params"].copy()
cb_w = CatBoostClassifier(
    task_type="GPU", devices="0",
    iterations=1700, od_type="Iter", od_wait=170,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=RANDOM_STATE, verbose=False,
    class_weights=[1.0, w_pos],
    **p
)
cb_w.fit(X_train, y_train, eval_set=(X_valid, y_valid), cat_features=cat_idx)
pred_w = cb_w.predict_proba(X_valid)[:, 1]
score_w = ing_hubs_datathon_metric(y_valid, pred_w)
print(f"🎯 ClassWeights | ING={score_w:.5f} (w_pos={w_pos:.2f})")

# Hangisi yüksekse onu al
if score_w > best["score"]:
    best["score"] = score_w
    best["model"] = cb_w
    best["params"]["class_weights"] = [1.0, w_pos]


In [ ]:
# Validation raporu
y_valid_pred = best["model"].predict_proba(X_valid)[:, 1]
_ = eval_ing(y_valid, y_valid_pred, label="FINAL (Valid) | ")

# Full fit (train+valid -> X,y)
final_params = best["params"].copy()
cb_final = CatBoostClassifier(
    task_type="GPU", devices="0",
    iterations=2200, od_type="Iter", od_wait=220,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=RANDOM_STATE, verbose=200,
    **final_params
)
cb_final.fit(X, y, cat_features=[X.columns.get_loc(c) for c in cat_cols if c in X.columns])

# Submission
X_test = test_df.drop(columns=DROP, errors="ignore")
test_pred = cb_final.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"cust_id": test_df["cust_id"], "churn": test_pred})
submission.to_csv("submission_catboost_ultra.csv", index=False)
print("📤 submission_catboost_ultra.csv yazıldı.")
